# Result personalization with text embedding models in Elasticsearch
This notebook demonstrates:
1. Loading fashion dataset
2. Index in Elasticsearch using image search
3. Search items with a search term
4. Simulate click data
5. Generate a user personalization vector from user behavior
6. Personalize results

Check out our [blog post](https://www.elastic.co/search-labs/blog/diversify-results-maximum-marginal-relevance) on this topic to learn more about 

## 1. Setup and Dependencies

In [1]:
!pip install -r requirements.txt


[notice] A new release of pip is available: 24.2 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import json
import requests
import numpy as np
import kagglehub
from itertools import repeat
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm
import time
from elasticsearch import Elasticsearch
from IPython.display import HTML, display
from typing import List, Dict, Tuple
import base64
from io import BytesIO

## 2. Load Configuration

Create a configuration file `elastic_config.env` in this format to authenticate with JINA and the Elastic Cluster. 
```
ELASTIC_API_KEY=<ELASTIC_KEY>
ELASTIC_HOST=<HOST_URL>
JINA_API_KEY=<JINA_KEY>
```

In [3]:
def load_config(file_path="elastic_config.env"):
    """Load configuration from environment file"""
    config = {}
    try:
        with open(file_path, "r") as file:
            for line in file:
                if "=" in line:
                    key, value = line.strip().split("=", 1)
                    config[key] = value
    except FileNotFoundError:
        print(f"Configuration file not found: {file_path}")
    return config


config = load_config()
elastic_host = config.get("ELASTIC_HOST")
elastic_api_key = config.get("ELASTIC_API_KEY")
jina_api_key = config.get("JINA_API_KEY")

print("Configuration loaded successfully")

Configuration loaded successfully


## 3. Load Dataset and Extract ID & Image URLs

In [4]:
dataset_path = kagglehub.dataset_download(
    "paramaggarwal/fashion-product-images-dataset"
)
print("Path to dataset files:", dataset_path)

styles_folder = os.path.join(dataset_path, "fashion-dataset/styles")


def load_dataset(folder_path):
    """Load all JSON files from the dataset folder"""
    products = []

    for filename in os.listdir(folder_path):
        if filename.endswith(".json"):
            file_path = os.path.join(folder_path, filename)
            try:
                with open(file_path, "r") as f:
                    data = json.load(f)
                    if "data" in data:
                        products.append(data["data"])
            except Exception as e:
                print(f"Error reading {filename}: {e}")

    return products


products = load_dataset(styles_folder)
print(f"Loaded {len(products)} total products")

# Filter for bottomwear only to limit data for this demo
filtered_products = []
for product in products:
    sub_category = product.get("subCategory", {})
    if "topwear" == sub_category.get("typeName", "").lower():
        filtered_products.append(product)

print(f"\nFiltered to {len(filtered_products)} products")

products = filtered_products

Path to dataset files: /Users/peter/.cache/kagglehub/datasets/paramaggarwal/fashion-product-images-dataset/versions/1
Loaded 44446 total products

Filtered to 15405 products


In [5]:
def extract_id_and_image_url(products):
    """Extract ID and image URL from products"""
    image_data = []

    for product in products:
        product_id = product.get("id")

        style_images = product.get("styleImages", {})
        default_image = style_images.get("default", {})

        image_url = default_image.get("resolutions", {}).get("360X480", "")
        if not image_url:
            image_url = default_image.get("imageURL", "")

        if product_id and image_url:
            image_data.append(
                {
                    "id": product_id,
                    "image_url": image_url,
                    "product_name": product.get("productDisplayName", ""),
                    "brand": product.get("brandName", ""),
                    "color": product.get("baseColour", ""),
                    "article_type": product.get("articleType", {}).get("typeName", ""),
                }
            )

    return image_data


image_data = extract_id_and_image_url(products)
print(f"Extracted {len(image_data)} products with valid IDs and image URLs")

# Only use 1000 products to not make the demo too heavy
demo_image_data = image_data[:1000]
print(f"\nLimited to {len(demo_image_data)} items for demo")
print(f"\nSample items (alphabetically sorted):")
for i in range(min(5, len(demo_image_data))):
    item = demo_image_data[i]
    print(f"  - {item['product_name']} ({item['article_type']}, {item['color']})")

Extracted 15401 products with valid IDs and image URLs

Limited to 1000 items for demo

Sample items (alphabetically sorted):
  - Wrangler Men Miriam Checks Pink Shirt (Shirts, Pink)
  - Flying Machine Women Black T-shirt (Tshirts, Black)
  - Palm Tree Kids Boys Printed Orange Tshirts (Tshirts, Orange)
  - Highlander Men Check Grey Shirt (Shirts, Grey)
  - Span Women Maroon & Black Kurta (Kurtas, Maroon)


## 4. Create Image Embeddings with JINA API

In [6]:
import json


def get_single_image_embedding(item, jina_api_key):
    """Get embedding for a single image"""
    url = "https://api.jina.ai/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {jina_api_key}",
    }

    product_data = {
        "product_name": item["product_name"],
        "brand": item["brand"],
        "color": item["color"],
        "article_type": item["article_type"],
    }

    data = {
        "model": "jina-embeddings-v4",
        "dimensions": 2048,
        "task": "retrieval.passage",
        "embedding_type": "float",
        "input": [{"text": f"{product_data}"}, {"image": item["image_url"]}],
    }

    try:
        response = requests.post(url, headers=headers, json=data, timeout=200)
        response.raise_for_status()

        result = response.json()
        if "data" in result and len(result["data"]) > 0:
            return {
                "id": item["id"],
                "image_url": item["image_url"],
                "product_name": item["product_name"],
                "brand": item["brand"],
                "color": item["color"],
                "article_type": item["article_type"],
                "image_vector": to_avg_vector(
                    [result["data"][0]["embedding"], result["data"][1]["embedding"]]
                ),
            }
        return None
    except Exception as e:
        print(f"Error processing {item}: {e}")
        return None


# encode image and product information in one vector
def to_avg_vector(vectors):
    vectors_array = np.array(vectors)

    avg_vector = np.mean(vectors_array, axis=0)

    norm = np.linalg.norm(avg_vector)
    if norm > 0:
        normalized_avg_vector = avg_vector / norm
    else:
        normalized_avg_vector = avg_vector

    return normalized_avg_vector.tolist()


print("Getting embeddings...")

with ThreadPoolExecutor(max_workers=10) as executor:
    products_with_vectors = list(
        tqdm(
            executor.map(
                get_single_image_embedding, demo_image_data, repeat(jina_api_key)
            ),
            total=len(demo_image_data),
            desc="Getting embeddings",
        )
    )

print(f"Retrieved {len(products_with_vectors)} embeddings")

Getting embeddings...


Getting embeddings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [20:30<00:00,  1.23s/it]

Retrieved 1000 embeddings


## 5. Setup Elasticsearch Index

In [7]:
es = Elasticsearch(elastic_host, api_key=elastic_api_key)

index_name = "fashion_products"

mapping = {
    "settings": {
        "index.mapping.exclude_source_vectors": False  # not recommended for production
    },
    "mappings": {
        "properties": {
            "id": {"type": "keyword"},
            "image_url": {"type": "keyword"},
            "product_name": {"type": "keyword"},
            "brand": {"type": "keyword"},
            "color": {"type": "keyword"},
            "article_type": {"type": "keyword"},
            "image_vector": {
                "type": "dense_vector",
                "dims": 2048,
                "index": True,
                "similarity": "dot_product",
                "index_options": {"type": "flat"},
            },
        }
    },
}

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)
    print(f"Deleted existing index '{index_name}'")

es.indices.create(index=index_name, body=mapping)
print(f"Created index '{index_name}'")

Deleted existing index 'fashion_products'
Created index 'fashion_products'


## 6. Index Documents with Image Vectors

In [8]:
def index_single_image(item):
    try:
        es.index(index=index_name, id=item["id"], document=item)
        return 1
    except Exception as e:
        print(f"Error indexing document {item['id']}: {e}")
        return 0


print("start")

# Index the documents in parallel
with ThreadPoolExecutor(max_workers=10) as executor:
    results = list(
        tqdm(
            executor.map(index_single_image, products_with_vectors),
            total=len(products_with_vectors),
            desc="Indexing images",
        )
    )

indexed_count = sum(results)
print(f"Successfully indexed {indexed_count} documents")

start


Indexing images: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [00:41<00:00, 24.29it/s]

Successfully indexed 1000 documents


## 7. Query Images with Text Search

In [13]:
SEARCH_QUERY = "workout shirt"


def get_text_embedding(text, jina_api_key):
    """Get text embedding from JINA API"""
    url = "https://api.jina.ai/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {jina_api_key}",
    }

    data = {
        "model": "jina-embeddings-v4",
        "dimensions": 2048,
        "embedding_type": "float",
        "task": "retrieval.query",
        "input": [{"text": text}],
    }

    try:
        response = requests.post(url, headers=headers, json=data, timeout=30)
        response.raise_for_status()
        result = response.json()

        if "data" in result and len(result["data"]) > 0:
            return result["data"][0]["embedding"]
    except Exception as e:
        print(f"Error getting text embedding: {e}")

    return None


def search_similar_images(es, index_name, query_vector, k=20):
    """Search for similar images using vector similarity"""
    query = {
        "knn": {
            "field": "image_vector",
            "query_vector": query_vector,
            "k": k,
        },
        "size": k,
    }

    response = es.search(index=index_name, body=query)

    results = []
    for hit in response["hits"]["hits"]:
        # Find the original product data to get additional info
        product_id = hit["_source"]["id"]

        results.append(
            {
                "id": product_id,
                "image_url": hit["_source"]["image_url"],
                "score": hit["_score"],
                "product_name": hit["_source"]["product_name"],
                "brand": hit["_source"]["brand"],
                "color": hit["_source"]["color"],
                "article_type": hit["_source"]["article_type"],
            }
        )

    return results


print(f"Creating text embedding for: '{SEARCH_QUERY}'")
query_vector = get_text_embedding(SEARCH_QUERY, jina_api_key)

if query_vector:
    print(f"\nSearching for items similar to: '{SEARCH_QUERY}'")
    search_results = search_similar_images(es, index_name, query_vector, k=150)
    print(f"Found {len(search_results)} similar images")
else:
    print("Failed to get text embedding")

Creating text embedding for: 'workout shirt'

Searching for items similar to: 'workout shirt'
Found 150 similar images


## 8. Display Search Results 
Showing results for text search: **"pants"**

In [14]:
def display_images(images, title="Images", max_per_row=5):
    """Display images in a grid layout"""
    html = f"<h2>{title}</h2>"
    html += '<div style="display: flex; flex-wrap: wrap; gap: 10px;">'
    images = images[:10]

    for i, img in enumerate(images):
        score = img.get("score", "N/A")
        if isinstance(score, (int, float)):
            score_str = f"{score:.3f}"
        else:
            score_str = "N/A"

        product_name = img.get("product_name", "N/A")
        if product_name != "N/A" and len(product_name) > 25:
            product_name = product_name[:25] + "..."

        html += f"""
       <div style="text-align: center; margin-bottom: 20px;">
           <img src="{img['image_url']}" style="width: 150px; height: 200px; object-fit: cover; border: 1px solid #ddd;">
           <p style="margin: 5px 0; font-size: 12px; font-weight: bold;">ID: {img['id']}</p>
           <p style="margin: 5px 0; font-size: 11px;" alit="{img.get('product_name', 'N/A')}">{product_name}</p>
           <p style="margin: 5px 0; font-size: 11px; color: #666;">{img.get('article_type', '')} - {img.get('color', '')}</p>
           <p style="margin: 5px 0; font-size: 12px; color: #007bff;">Score: {score_str}</p>
       </div>
       """

        if (i + 1) % max_per_row == 0:
            html += '</div><div style="display: flex; flex-wrap: wrap; gap: 10px;">'

    html += "</div>"
    display(HTML(html))


display_images(search_results, "Original Search Results")

## 9. Generate click data

In [23]:
click_index_name = "clicks"

user_id = "person123"

clicks = [
    {
        "user_id": user_id,
        "product_id": "7867",
        "image_url": "http://assets.myntassets.com/v1/images/style/properties/a4a112238eab67ac7d2a3e5e3222bb7f_images.jpg",
        "timestamp": "2025-10-07T09:04:45.565503",
    },
    {
        "user_id": user_id,
        "product_id": "8403",
        "image_url": "http://assets.myntassets.com/v1/images/style/properties/8d742d940bb052f421a666ff2157214a_images.jpg",
        "timestamp": "2025-10-07T09:15:12.817652",
    },
    {
        "user_id": user_id,
        "product_id": "7563",
        "image_url": "http://assets.myntassets.com/v1/images/style/properties/b1d562e2d87fbf047d98230fb35b273c_images.jpg",
        "timestamp": "2025-10-07T09:04:54.142514",
    },
]

click_mapping = {
    "settings": {"index": {"mode": "lookup"}},
    "mappings": {
        "properties": {
            "user_id": {"type": "keyword"},
            "product_id": {"type": "keyword"},
            "image_url": {"type": "keyword"},
            "timestamp": {"type": "date"},
        }
    },
}

if es.indices.exists(index=click_index_name):
    es.indices.delete(index=click_index_name)
es.indices.create(index=click_index_name, body=click_mapping)


print(f"Indexing {len(clicks)} click data points...")
for i, click_data in enumerate(clicks):
    try:
        es.index(index=click_index_name, document=click_data)
        print(
            f"  Indexed click {i+1}: user={click_data['user_id']}, product={click_data['product_id']}"
        )
    except Exception as e:
        print(f"  Error indexing click {i+1}: {e}")

print(f"\nSuccessfully indexed {len(clicks)} click data points")

es.indices.refresh(index=click_index_name)
count = es.count(index=click_index_name)
print(f"Total documents in {click_index_name}: {count['count']}")

Indexing 3 click data points...
  Indexed click 1: user=person123, product=7867
  Indexed click 2: user=person123, product=8403
  Indexed click 3: user=person123, product=7563

Successfully indexed 3 click data points
Total documents in clicks: 3


## 10. Create Personalization vector
### 10.1 Get all clicks from the clicks index
### 10.2 Get the vectors for the clicked products
### 10.3 Calculate the average of all clicked vectors
(To take this further other types such as purchase or add-to-cart events can be considered and a weighted averaged can be created)

In [24]:
def get_user_personalization_vector(user_id, click_index, product_index):
    """
    Generate a personalization vector for a user based on their click history.

    1. Get all clicks for the user
    2. Retrieve product vectors for clicked products
    3. Average the vectors to create a personalization vector
    """

    print(f"Fetching click history for user: {user_id}")

    click_query = {
        "query": {"match": {"user_id": user_id}},
        "size": 10000,  # Get all clicks (adjust if needed)
    }

    click_response = es.search(index=click_index, body=click_query)
    clicks = click_response["hits"]["hits"]

    if not clicks:
        print(f"No clicks found for user {user_id}")
        return None

    print(f"Found {len(clicks)} clicks for user {user_id}")

    clicked_product_ids = list(set([hit["_source"]["product_id"] for hit in clicks]))
    print(f"Unique products clicked: {len(clicked_product_ids)}")
    print(f"Product IDs: {clicked_product_ids}")

    print("\nRetrieving product vectors...")

    products_query = {
        "_source": ["image_vector"],
        "query": {"ids": {"values": clicked_product_ids}},
        "size": 10,
    }

    product_vectors = []
    try:
        products_response = es.search(index=product_index, body=products_query)

        for hit in products_response["hits"]["hits"]:
            product_vectors.append(hit["_source"]["image_vector"])

    except Exception as e:
        print(f"  ✗ Error retrieving products: {e}")

    print(f"Successfully retrieved {len(product_vectors)} product vectors")

    print("Creating personalization vector...")

    vectors_array = np.array(product_vectors)

    # Calculate the mean vector
    personalization_vector = np.mean(vectors_array, axis=0)

    # Normalize the vector (important for cosine similarity)
    norm = np.linalg.norm(personalization_vector)
    if norm > 0:
        personalization_vector = personalization_vector / norm

    print(
        f"Generated personalization vector with dimension: {len(personalization_vector)}"
    )

    return personalization_vector.tolist()


# Generate personalization vector for person123
user_id = "person123"
personalization_vector = get_user_personalization_vector(
    user_id=user_id, click_index=click_index_name, product_index=index_name
)

if personalization_vector:
    print(f"\n✅ Successfully created personalization vector for {user_id}")
    print(f"Vector dimension: {len(personalization_vector)}")
    print(f"Vector sample (first 5 values): {personalization_vector[:5]}")
else:
    print(f"\n❌ Failed to create personalization vector for {user_id}")

Fetching click history for user: person123
Found 3 clicks for user person123
Unique products clicked: 3
Product IDs: ['7563', '8403', '7867']

Retrieving product vectors...
Successfully retrieved 3 product vectors
Creating personalization vector...
Generated personalization vector with dimension: 2048

✅ Successfully created personalization vector for person123
Vector dimension: 2048
Vector sample (first 5 values): [0.013954155488779353, -0.01211520505902334, 0.04489586338920903, 0.0005054446129302575, -0.028102111160044244]


## 11. Search with personalization rescore

In [25]:
def search_similar_images_with_personalization(
    es, index_name, query_vector, personalization_vector, k=20
):
    """Search for similar images using vector similarity"""
    query = {
        "retriever": {
            "rescorer": {
                "rescore": {
                    "window_size": k,
                    "query": {
                        "rescore_query": {
                            "script_score": {
                                "query": {"match_all": {}},
                                "script": {
                                    "source": "cosineSimilarity(params.queryVector, 'image_vector') + 1.0",
                                    "params": {"queryVector": personalization_vector},
                                },
                            }
                        }
                    },
                },
                "retriever": {
                    "knn": {
                        "field": "image_vector",
                        "query_vector": query_vector,
                        "k": k,
                        "num_candidates": k * 2,
                    }
                },
            }
        },
        "size": k,
    }

    response = es.search(index=index_name, body=query)

    results = []
    for hit in response["hits"]["hits"]:
        # Find the original product data to get additional info
        product_id = hit["_source"]["id"]

        results.append(
            {
                "id": product_id,
                "image_url": hit["_source"]["image_url"],
                "score": hit["_score"],
                "product_name": hit["_source"]["product_name"],
                "brand": hit["_source"]["brand"],
                "color": hit["_source"]["color"],
                "article_type": hit["_source"]["article_type"],
            }
        )

    return results


if query_vector:
    print(
        f"\nSearching for items similar to: '{SEARCH_QUERY}' with personal preference"
    )
    personalized_search_results = search_similar_images_with_personalization(
        es, index_name, query_vector, personalization_vector, k=150
    )
    print(f"Found {len(search_results)} similar images")
else:
    print("Failed to get text embedding")


Searching for items similar to: 'workout shirt' with personal preference
Found 150 similar images


In [26]:
display_images(personalized_search_results, "Original Search Results")